In [1]:
import random
import math
import pandas as pd
import numpy as np
import time


from preferences import prefs
from user import user_prefs, experiments

startPoint = [59.927085, 30.317504]
endPoint = [59.935408, 30.327108]

In [2]:
df = pd.read_csv('data/places.csv')

In [3]:
POPULATION_SIZE = 100
GENERATIONS = 100

MIN_ROUTE_POINTS = 2
MAX_ROUTE_POINTS = 15

MUTATION_RATE = 0.2
TOURNAMENT_SIZE = 5

def point_interest_score(point_row):
    score = 0
    for category, tags in prefs.items():
        weight = user_prefs[category]
        for tag in tags:
            if tag in point_row.index and point_row[tag]:
                score += weight

    return score


def crowding_distance(front):

    n = len(front)

    if n == 0:
        return {}

    distance = [0.0 for _ in range(n)]

    num_obj = len(front[0]["fitness"])

    for m in range(num_obj):

        idx = list(range(n))

        idx.sort(key=lambda i: front[i]["fitness"][m])

        distance[idx[0]] = float("inf")
        distance[idx[-1]] = float("inf")

        min_f = front[idx[0]]["fitness"][m]
        max_f = front[idx[-1]]["fitness"][m]

        if max_f == min_f:
            continue

        for i in range(1, n - 1):

            prev_f = front[idx[i - 1]]["fitness"][m]
            next_f = front[idx[i + 1]]["fitness"][m]

            distance[idx[i]] += (
                (next_f - prev_f) / (max_f - min_f)
            )

    return distance


# Расстояние между точками в метрах
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000

    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)

    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    a = (
        math.sin(dphi / 2) ** 2
        + math.cos(phi1)
        * math.cos(phi2)
        * math.sin(dlambda / 2) ** 2
    )

    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

import numpy as np

def build_distance_matrix(df):

    n = len(df)
    dist = np.zeros((n, n))

    for i in range(n):
        for j in range(n):

            if i == j:
                continue

            dist[i][j] = haversine(
                df.iloc[i]["lat"],
                df.iloc[i]["lon"],
                df.iloc[j]["lat"],
                df.iloc[j]["lon"]
            )

    return dist

def init_pheromones(n):

    return np.ones((n, n))

def heuristic(i, j, df, dist):

    interest = point_interest_score(df.iloc[j])

    return interest / (dist[i][j] + 1e-6)

    import random

def select_next(current, visited, pheromones, dist, df, alpha=1, beta=2):

    n = len(df)

    probs = []

    for j in range(n):

        if j in visited:
            continue

        tau = pheromones[current][j] ** alpha
        eta = heuristic(current, j, df, dist) ** beta

        probs.append((j, tau * eta))

    if not probs:
        return None

    total = sum(p for _, p in probs)

    probs = [(j, p / total) for j, p in probs]

    r = random.random()
    cumulative = 0

    for j, p in probs:

        cumulative += p

        if r <= cumulative:
            return j

    return probs[-1][0]

def build_ant_route(start_idx, df, pheromones, dist, max_len=15):

    route = [start_idx]
    visited = set(route)

    current = start_idx

    for _ in range(max_len - 1):

        next_node = select_next(
            current,
            visited,
            pheromones,
            dist,
            df
        )

        if next_node is None:
            break

        route.append(next_node)
        visited.add(next_node)
        current = next_node

    return route

def update_pheromones(pheromones, ants, df, decay=0.5):

    pheromones *= (1 - decay)

    for ant in ants:

        route = ant["route"]
        score = ant["score"]

        for i in range(len(route) - 1):

            a = route[i]
            b = route[i + 1]

            pheromones[a][b] += score
            pheromones[b][a] += score

    return pheromones


def ant_colony(df, start_idx, n_ants=30, iterations=50):

    n = len(df)

    dist = build_distance_matrix(df)
    pheromones = init_pheromones(n)

    best_route = None
    best_score = -1

    for it in range(iterations):

        ants = []

        for _ in range(n_ants):

            route = build_ant_route(
                start_idx,
                df,
                pheromones,
                dist
            )

            score = sum(
                point_interest_score(df.iloc[i])
                for i in route
            )

            ants.append({
                "route": route,
                "score": score
            })

        pheromones = update_pheromones(
            pheromones,
            ants,
            df
        )

        best_ant = max(ants, key=lambda x: x["score"])

        if best_ant["score"] > best_score:
            best_score = best_ant["score"]
            best_route = best_ant["route"]

        print(f"Iter {it}: best = {best_score}")

    return best_route

In [4]:
# Полный маршрут
def project_point_to_line(point, start, end):
    """
    Проекция точки на линию START -> END.
    Возвращает параметр t:
    0   = начало линии
    1   = конец линии
    >1  = дальше конца
    <0  = до начала
    """

    px, py = point
    sx, sy = start
    ex, ey = end

    line_vec = np.array([ex - sx, ey - sy])
    point_vec = np.array([px - sx, py - sy])

    line_len_sq = np.dot(line_vec, line_vec)

    if line_len_sq == 0:
        return 0

    t = np.dot(point_vec, line_vec) / line_len_sq

    return t

def build_full_route(best_individual, df):

    route_points = []

    # ==========================================
    # Получаем точки маршрута
    # ==========================================

    points = []

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        lat = float(point["lat"])
        lon = float(point["lon"])

        # --------------------------------------
        # Позиция вдоль линии START -> END
        # --------------------------------------

        t = project_point_to_line(
            [lat, lon],
            startPoint,
            endPoint
        )

        points.append({
            "name": point["name"],
            "lat": lat,
            "lon": lon,
            "t": t
        })

    # ==========================================
    # Сортировка по географическому порядку
    # ==========================================

    points.sort(key=lambda x: x["t"])

    # ==========================================
    # Финальный маршрут
    # ==========================================

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    route_points.extend(points)

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

    route_points = []

    route_points.append({
        "name": "START",
        "lat": startPoint[0],
        "lon": startPoint[1]
    })

    for idx in best_individual["route"]:

        point = df.iloc[idx]

        route_points.append({
            "name": point["name"],
            "lat": point["lat"],
            "lon": point["lon"]
        })

    route_points.append({
        "name": "END",
        "lat": endPoint[0],
        "lon": endPoint[1]
    })

    return route_points

# Ссылка на яндекс карты
def generate_yandex_maps_url(route_points):
    """
    route_points:
    [
        {"name": "...", "lat": ..., "lon": ...},
        ...
    ]
    """

    rtext = "~".join(
        f"{point['lat']},{point['lon']}"
        for point in route_points
    )

    yandex_url = (
        f"https://yandex.ru/maps/"
        f"?mode=routes"
        f"&rtext={rtext}"
        f"&rtt=pd"
    )

    return yandex_url

In [5]:
import time
import pandas as pd

# =========================================================
# Метрики
# =========================================================

def route_interest_score(route, df):

    total = 0

    for idx in route:
        point = df.iloc[idx]
        total += point_interest_score(point)

    return total


def diversity_bonus(route, df):

    used_categories = set()

    for idx in route:

        point = df.iloc[idx]

        for category_name, tags in prefs.items():

            for tag in tags:

                if tag in point and point[tag]:
                    used_categories.add(category_name)

    return len(used_categories) * 15


def route_smoothness_penalty(route, df):

    if len(route) < 3:
        return 0

    penalty = 0
    prev_direction = None

    points = [startPoint]

    for idx in route:

        point = df.iloc[idx]
        points.append([float(point["lat"]), float(point["lon"])])

    points.append(endPoint)

    for i in range(len(points) - 1):

        p1 = points[i]
        p2 = points[i + 1]

        direction = (
            p2[0] - p1[0],
            p2[1] - p1[1]
        )

        if prev_direction is not None:

            dot = (
                direction[0] * prev_direction[0] +
                direction[1] * prev_direction[1]
            )

            if dot < 0:
                penalty += 50

        prev_direction = direction

    return penalty


# =========================================================
# Выбор лучшего решения из Pareto
# =========================================================

def choose_best_solution(pareto_front, df, user_prefs):

    best = None
    best_score = -float("inf")

    for sol in pareto_front:

        route = sol["route"]

        interest = route_interest_score(route, df)
        distance = route_distance(route, df)

        score = (
            interest * 1.0
            - distance * 0.001
            + diversity_bonus(route, df)
            - route_smoothness_penalty(route, df)
        )

        # пользовательский bias
        score *= (1 + sum(user_prefs.values()) / 20)

        if score > best_score:
            best_score = score
            best = sol

    return best


# =========================================================
# Один эксперимент
# =========================================================

def run_experiment(df, user_prefs, experiment_name="exp"):

    start_time = time.time()

    # =====================================================
    # ACO вместо NSGA-II
    # =====================================================

    start_idx = 0  # можно заменить на ближайшую точку к startPoint

    best_route = ant_colony(
        df=df,
        start_idx=start_idx,
        n_ants=30,
        iterations=50
    )

    # =====================================================
    # финальный маршрут
    # =====================================================

    best_individual = {
        "route": best_route
    }

    final_route = build_full_route(best_individual, df)

    end_time = time.time()

    # =====================================================
    # метрики
    # =====================================================

    interest = route_interest_score(best_route, df)
    distance = route_distance(best_route, df)
    duration = end_time - start_time

    yandex_url = generate_yandex_maps_url(final_route)

    route_text = route_to_names(final_route)

    return {
        "experiment": experiment_name,
        "interest_score": interest,
        "distance": distance,
        "time_sec": duration,
        "route_length": len(best_route),
        "route_text": route_text,
        "yandex_url": yandex_url
    }

# =========================================================
# Множественные эксперименты
# =========================================================

def run_multiple_experiments(df, experiments):

    results = []

    for exp in experiments:

        print(f"\nRunning: {exp['name']}")

        result = run_experiment(
            df=df,
            user_prefs=exp["user_prefs"],
            experiment_name=exp["name"]
        )

        results.append(result)

        print(
            f"Done: {exp['name']} "
            f"| score: {result['interest_score']:.2f} "
            f"| time: {result['time_sec']:.2f}s"
        )

    return results

# =========================================================
# Названия маршрута
# =========================================================

def route_to_names(final_route):

    names = []

    for point in final_route:

        if point["name"] not in ["START", "END"]:
            names.append(point["name"])

    return " -> ".join(names)


# =========================================================
# Сохранение
# =========================================================

def save_results(results, filename="experiments.csv"):

    df_results = pd.DataFrame(results)
    df_results.to_csv(filename, index=False)

    print(f"\nSaved to {filename}")


# =========================================================
# Запуск
# =========================================================

results = run_multiple_experiments(df, experiments)

save_results(results)


Running: user_1_military_focus


KeyboardInterrupt: 